In [48]:
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score
from sklearn.preprocessing import StandardScaler
from utils import custom_score
from pathlib import Path
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

In [3]:
data_dir = Path('../data')
X = pd.read_csv(data_dir / 'x_train.txt', sep=' ')
y = pd.read_csv(data_dir / 'y_train.txt', sep=' ').values.ravel()

feature_dir = Path("../feature_selection/selected_features.txt")
with open(feature_dir, "r") as f:
    content = f.read()
selected_features = [x.strip().strip("'").strip('"') for x in content.split(",")]

X = X[selected_features]

In [4]:
data_dir = Path('../data')
X = pd.read_csv(data_dir / 'x_train.txt', sep=' ')
y = pd.read_csv(data_dir / 'y_train.txt', sep=' ').values.ravel()

feature_dir = Path("../feature_selection/selected_features.txt")
with open(feature_dir, "r") as f:
    content = f.read()
selected_features = [x.strip().strip("'").strip('"') for x in content.split(",")]

X = X[selected_features]
# ── Helper: evaluate one (feature subset, hyperparams) config via CV ──────────
def evaluate_config(X, y, features, params, k=5, max_contacts=1000):
    """
    Returns mean project score across k folds.
    features: which columns of X to use
    params: dict of random forest hyperparameters
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    scores = []
    n_var = len(features)

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]
        # tak z ciekawosci
        print(f"{accuracy_score(y_val, probs>1/3)}")

        # Decision rule: target customers above threshold, cap at proportional max_contacts
        # (scaled to fold size, since each fold is ~20% of training data)
        fold_cap = int(max_contacts * len(val_idx) / len(y))
        threshold = 1/3  # expected-value break-even: 10p - 5(1-p) > 0
        
        # Take customers with p > threshold, then cap at fold_cap
        candidates = np.where(probs > threshold)[0]
        if len(candidates) > fold_cap:
            top = candidates[np.argsort(-probs[candidates])[:fold_cap]]
        else:
            top = candidates

        y_pred = np.zeros_like(y_val)
        y_pred[top] = 1
        scores.append(custom_score(y_val, y_pred, n_var))

    return np.mean(scores)

# ── Hyperparameter grid  ──────────────────────────────────
param_grid = [
    {"n_estimators": 200, "n_jobs": -1},
    {"n_estimators": 500, "n_jobs": -1},
    {"n_estimators": 700, "n_jobs": -1},
]

# ── Forward selection driven by project score ─────────────────────────────────
n_features = X.shape[1]
selected = ['V11', 'V176', 'V191', 'V255', 'V309']
remaining = list(set(selected_features) - set(selected))
best_score = -np.inf
best_params = None
history = []

while remaining:
    fold_best_score = -np.inf
    fold_best_feat = None
    fold_best_params = None

    for feat in remaining:
        trial_subset = selected + [feat]
        for params in param_grid:
            score = evaluate_config(X, y, trial_subset, params)
            if score > fold_best_score:
                fold_best_score = score
                fold_best_feat = feat
                fold_best_params = params

    # Stop if adding any feature hurts (the −200 penalty isn't paid off)
    if fold_best_score <= best_score:
        print(f"\nNo feature improves score. Stopping.")
        break

    selected.append(fold_best_feat)
    remaining.remove(fold_best_feat)
    best_score = fold_best_score
    best_params = fold_best_params
    history.append((len(selected), fold_best_feat, fold_best_score, fold_best_params))
    print(f"Step {len(selected)}: added feat {fold_best_feat} | "
          f"score = {fold_best_score} | params = {fold_best_params}")

print(f"\n── Final ──")
print(f"Selected {len(selected)} features: {selected}")
print(f"Best params: {best_params}")
print(f"CV score:    {best_score}")

0.539
0.55
0.557
0.549
0.566
0.539
0.543
0.558
0.549
0.564
0.541
0.546
0.56
0.545
0.568
0.533
0.537
0.553
0.54
0.552
0.538
0.547
0.553
0.531
0.552
0.538


KeyboardInterrupt: 

In [7]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

data_dir = Path('../data')
X = pd.read_csv(data_dir / 'x_train.txt', sep=' ')
y = pd.read_csv(data_dir / 'y_train.txt', sep=' ').values.ravel()

feature_dir = Path("../feature_selection/selected_features.txt")
with open(feature_dir, "r") as f:
    content = f.read()

selected_features = [x.strip().strip("'").strip('"') for x in content.split(",")]
X = X[selected_features]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, objective='binary:logistic')
xgb.fit(X_train, y_train)
pred = xgb.predict_proba(X_test)
print(accuracy_score(y_test, pred[:,1]>1/3))
print(custom_score(y_test, pred[:,1]>1/3, len(X_train.columns)))

0.623
-2940


In [11]:
# do xgb trzeba ustawic wiecej seedow
SEED = 3
# Python randomness
random.seed(SEED)
# NumPy randomness
np.random.seed(SEED)
keep = ['V11', 'V176', 'V191', 'V255', 'V309']
X = X[keep]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, objective='binary:logistic',random_state=SEED, seed=SEED)
xgb.fit(X_train, y_train)
pred = xgb.predict_proba(X_test)
print(accuracy_score(y_test, pred[:,1]>1/3))
print(custom_score(y_test, pred[:,1]>1/3, X_train.shape[1]))

0.57
1240


In [9]:
# do xgb trzeba ustawic wiecej seedow
SEED = 3
# Python randomness
random.seed(SEED)
# NumPy randomness
np.random.seed(SEED)
keep = ['V11', 'V176', 'V191', 'V255', 'V309']
X = X[keep]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, objective='binary:logistic',random_state=SEED, seed=SEED)
xgb.fit(X_train, y_train)
pred = xgb.predict_proba(X_test)
print(accuracy_score(y_test, pred[:,1]>1/3))
print(custom_score(y_test, pred[:,1]>1/3, X_train.shape[1]))

0.57
1240


## models comparison

In [ ]:
def sweep_threshold_and_k(thresh_probs, y_thresh, thresholds_grid, k_grid, n_vars):
    """
    Przeszukuje siatkę progów i limitów K, zwracając kombinację dającą najwyższy custom_score.
    """
    best_thr = 1/3
    best_k = 150
    best_score = -np.inf
    thresholds = np.linspace(0.05, 0.65, 80)
    k_contacts_grid = np.arange(50, 801, 20)
    
    for t in thresholds:
        for k_cap in k_contacts_grid:
            # Wybieramy indeksy spełniające próg prawdopodobieństwa
            candidates = np.where(thresh_probs > t)[0]
            
            # Ograniczamy liczbę kontaktów do k_cap
            if len(candidates) > k_cap:
                top_idx = candidates[np.argsort(-thresh_probs[candidates])[:k_cap]]
            else:
                top_idx = candidates
            
            #  wektor predykcji
            preds = np.zeros_like(y_thresh)
            preds[top_idx] = 1
            
            score = custom_score(y_thresh, preds, n_vars)
            
            if score > best_score:
                best_score = score
                best_thr = t
                best_k = k_cap
                
    return best_thr, best_k, best_score


def apply_threshold(test_probs, threshold, max_k):
    """
    Aplikuje wyznaczony próg oraz limit kontaktów K na zbiorze testowym.
    """
    candidates = np.where(test_probs > threshold)[0]
    
    if len(candidates) > max_k:
        top_idx = candidates[np.argsort(-test_probs[candidates])[:max_k]]
    else:
        top_idx = candidates
        
    preds = np.zeros_like(test_probs, dtype=int)
    preds[top_idx] = 1
    return preds

In [ ]:
models_dict = {
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        random_state=SEED, verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED, 
        verbose=-1
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=6, random_state=SEED, n_jobs=-1
    ),
    'SVM_RBF': SVC(
        kernel='rbf', C=1.0, gamma='scale', 
        probability=True, random_state=SEED 
    ),
    'LogisticRegression': LogisticRegression(
        C=1.0,             
        l1_ratio = 0,
        solver='lbfgs',    
        max_iter=1000,       
        random_state=SEED
    ),
    'LogisticRegression_L1_Standard': LogisticRegression(
        C=1.0,              
        l1_ratio = 1.0,     #lasso
        solver='saga',       
        max_iter=5000,     
        random_state=SEED
    ),
    'LogisticRegression_L1_Strong': LogisticRegression(
        C=0.1,              
        l1_ratio = 1.0,   
        solver='saga',       
        max_iter=5000,       
        random_state=SEED
    )
}

In [57]:

N_VARS = X.shape[1] 
summary_results = []
outer_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for model_name, model_obj in models_dict.items():
    print(f"=== Rozpoczynam walidację dla modelu: {model_name} ===")
    
    # Listy na metryki z poszczególnych foldów dla danego modelu
    fold_scores = []
    fold_accuracies = []
    fold_precisions = []
    fold_thresholds = []
    fold_ks = []
    
    for fold_idx, (outer_train_idx, outer_test_idx) in enumerate(outer_skf.split(X, y)):
        
        X_outer_train = X.iloc[outer_train_idx]
        y_outer_train = y[outer_train_idx]
        X_outer_test  = X.iloc[outer_test_idx]
        y_outer_test  = y[outer_test_idx]
        
        # split for training the model, and for finding threshold
        X_pure_tr, X_thresh, y_pure_tr, y_thresh = train_test_split(
            X_outer_train, y_outer_train,
            test_size=0.20, stratify=y_outer_train, random_state=SEED
        )
        
        scaler = StandardScaler()
        scaler.set_output(transform="pandas")
        X_pure_tr_sc   = scaler.fit_transform(X_pure_tr)
        X_thresh_sc    = scaler.transform(X_thresh)
        X_outer_test_sc = scaler.transform(X_outer_test)
        
        # cloning the model, to make sure they are training from scratch
        fold_model = clone(model_obj)
        fold_model.fit(X_pure_tr_sc, y_pure_tr)
        
        # optimal threshold
        thresh_probs = fold_model.predict_proba(X_thresh_sc)[:, 1]
        best_thr, best_k, _ = sweep_threshold_and_k(
            thresh_probs, y_thresh, thresh_probs, y_thresh, N_VARS
        )
        
        # score on outer_test - scale K to test size
        scale = len(y_outer_test) / len(y_thresh)
        test_k = int(best_k * scale)
        
        # prediction on outer test with the found threshold and K
        test_probs = fold_model.predict_proba(X_outer_test_sc)[:, 1]
        y_pred = apply_threshold(test_probs, best_thr, max_k=test_k)
        
        score = custom_score(y_outer_test, y_pred, N_VARS)
        acc   = accuracy_score(y_outer_test, y_pred)
        # zero_division=0 secures for situations, when model for high threshold might predict no positives
        prec  = precision_score(y_outer_test, y_pred, zero_division=0)
        
        fold_scores.append(score)
        fold_accuracies.append(acc)
        fold_precisions.append(prec)
        fold_thresholds.append(best_thr)
        fold_ks.append(best_k)
        
        #print(f"  Fold {fold_idx+1}: Score={score:.1f} | Acc={acc:.3f} | Prec={prec:.3f} | thr={best_thr:.3f} | K={best_k}")
    
    # Obliczanie średnich i odchyleń standardowych dla modelu po 5 foldach
    summary_results.append({
        'Model': model_name,
        'Mean Custom Score': np.mean(fold_scores),
        'Std Custom Score': np.std(fold_scores),
        'Mean Accuracy': np.mean(fold_accuracies),
        'Mean Precision': np.mean(fold_precisions),
        'Mean Threshold': np.mean(fold_thresholds),
        'Mean K': np.mean(fold_ks)
    })
    print(f"=== Zakończono dla {model_name}. Mean Score: {np.mean(fold_scores):.1f} ===\n")

df_summary = pd.DataFrame(summary_results)
# sorting by custom score
df_summary = df_summary.sort_values(by='Mean Custom Score', ascending=False).reset_index(drop=True)

print(df_summary.to_string(index=False))

=== Rozpoczynam walidację dla modelu: XGBoost ===
=== Zakończono dla XGBoost. Mean Score: 1465.0 ===

=== Rozpoczynam walidację dla modelu: LightGBM ===
=== Zakończono dla LightGBM. Mean Score: 1427.0 ===

=== Rozpoczynam walidację dla modelu: RandomForest ===
=== Zakończono dla RandomForest. Mean Score: 1430.0 ===

=== Rozpoczynam walidację dla modelu: SVM_RBF ===
=== Zakończono dla SVM_RBF. Mean Score: 1466.0 ===

=== Rozpoczynam walidację dla modelu: LogisticRegression ===
=== Zakończono dla LogisticRegression. Mean Score: 1425.0 ===

=== Rozpoczynam walidację dla modelu: LogisticRegression_L1_Standard ===
=== Zakończono dla LogisticRegression_L1_Standard. Mean Score: 1425.0 ===

=== Rozpoczynam walidację dla modelu: LogisticRegression_L1_Strong ===
=== Zakończono dla LogisticRegression_L1_Strong. Mean Score: 1410.0 ===

                         Model  Mean Custom Score  Std Custom Score  Mean Accuracy  Mean Precision  Mean Threshold  Mean K
                       SVM_RBF           

In [ ]:
# mean threshold for final trening?
final_thr = np.mean(best_thresholds)
final_k  = int(np.mean(best_ks))

# To już chyba na koniec, ale nie wiem? Retrain na wszystkich 5000
scaler_final = StandardScaler()
X_all_sc = scaler_final.fit_transform(X)

final_model = XGBClassifier(**params)
final_model.fit(X_all_sc, y)

# Predykcja na x_test.txt
X_submission_sc = scaler_final.transform(X_test)
probs_submission = final_model.predict_proba(X_submission_sc)[:, 1]
top_indices = np.argsort(-probs_submission)[:final_k]

c:\Users\miktw\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
